In [1]:
# ============================================================
#  AZ-104 Data Lab — Real-World Data Cleaning & Preprocessing
#  Data Source : Open-Meteo Weather API (Lodi, NJ)
#  Libraries   : pandas, numpy, scikit-learn, missingno
#  Author      : Allen | Date: May 31, 2026
# ============================================================

import requests
import pandas as pd
import numpy as np
import missingno as msno
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# SECTION 1 — FETCH REAL-WORLD DATA FROM OPEN-METEO API
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("  STEP 1: Fetching live weather data for Lodi, NJ...")
print("=" * 60)

LAT = 40.8776
LON = -74.0826

url = (
    "https://api.open-meteo.com/v1/forecast"
    f"?latitude={LAT}&longitude={LON}"
    "&hourly=temperature_2m,relative_humidity_2m,"
    "precipitation,windspeed_10m,weathercode"
    "&timezone=America%2FNew_York"
    "&past_days=14&forecast_days=1"
)

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    raw = response.json()
    df = pd.DataFrame(raw["hourly"])
    print(f"  ✅ Data fetched! Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")
except Exception as e:
    print(f"  ⚠️  API call failed ({e}). Loading offline sample data...\n")
    np.random.seed(42)
    n = 72
    times = pd.date_range("2026-05-17", periods=n, freq="h")
    df = pd.DataFrame({
        "time"                 : times.strftime("%Y-%m-%dT%H:%M"),
        "temperature_2m"       : np.random.normal(18, 5, n),
        "relative_humidity_2m" : np.random.normal(65, 15, n),
        "precipitation"        : np.random.exponential(0.5, n),
        "windspeed_10m"        : np.random.normal(12, 4, n),
        "weathercode"          : np.random.choice([0,1,2,3,45,61,80], n),
    })

# ─────────────────────────────────────────────────────────────
# SECTION 2 — INITIAL EXPLORATION
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("  STEP 2: Initial Data Exploration")
print("=" * 60)

print(f"\n📐 Shape      : {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\n📋 Columns    : {list(df.columns)}")
print(f"\n🔢 Data Types :\n{df.dtypes}")
print(f"\n📊 First 5 rows:\n{df.head()}")
print(f"\n📈 Statistics :\n{df.describe()}")

# ─────────────────────────────────────────────────────────────
# SECTION 3 — INJECT MISSING VALUES (for practice)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 3: Injecting Missing Values for Practice")
print("=" * 60)

np.random.seed(7)
for col in ["temperature_2m","relative_humidity_2m",
            "precipitation","windspeed_10m"]:
    idx = np.random.choice(df.index, size=int(len(df)*0.08), replace=False)
    df.loc[idx, col] = np.nan

print(f"\n🔍 Missing values per column:\n{df.isnull().sum()}")
print(f"\n📉 Missing % per column:\n{round(df.isnull().mean()*100,2)}")

# ─────────────────────────────────────────────────────────────
# SECTION 4 — VISUALISE MISSING DATA (missingno)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 4: Visualising Missing Data with missingno")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
msno.bar(df, ax=axes[0], color="#2196F3", fontsize=11)
axes[0].set_title("Missing Data — Bar Chart", fontsize=13, fontweight="bold")
msno.matrix(df, ax=axes[1], sparkline=False, fontsize=11)
axes[1].set_title("Missing Data — Matrix", fontsize=13, fontweight="bold")
plt.suptitle("Lodi, NJ — Open-Meteo Weather (Missing Value Analysis)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("missing_data_analysis.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✅ Chart saved → missing_data_analysis.png")

# ─────────────────────────────────────────────────────────────
# SECTION 5 — IMPUTE MISSING VALUES
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 5: Handling Missing Values (Imputation)")
print("=" * 60)

mean_imputer = SimpleImputer(strategy="mean")
df[["temperature_2m","relative_humidity_2m"]] = mean_imputer.fit_transform(
    df[["temperature_2m","relative_humidity_2m"]])
print("  ✅ temperature_2m & relative_humidity_2m → imputed with MEAN")

median_imputer = SimpleImputer(strategy="median")
df[["precipitation","windspeed_10m"]] = median_imputer.fit_transform(
    df[["precipitation","windspeed_10m"]])
print("  ✅ precipitation & windspeed_10m         → imputed with MEDIAN")
print(f"\n  Missing values remaining: {df.isnull().sum().sum()}")

# ─────────────────────────────────────────────────────────────
# SECTION 6 — HANDLE OUTLIERS (IQR method)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 6: Detecting & Capping Outliers (IQR Method)")
print("=" * 60)

def cap_outliers(dataframe, column):
    Q1, Q3 = dataframe[column].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((dataframe[column] < lower)|(dataframe[column] > upper)).sum()
    dataframe[column] = dataframe[column].clip(lower=lower, upper=upper)
    print(f"  {column:<30} → {n_out:>3} outliers capped [{lower:.2f}, {upper:.2f}]")
    return dataframe

for col in ["temperature_2m","relative_humidity_2m",
            "precipitation","windspeed_10m"]:
    df = cap_outliers(df, col)

# ─────────────────────────────────────────────────────────────
# SECTION 7 — FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 7: Feature Engineering")
print("=" * 60)

df["time"]         = pd.to_datetime(df["time"])
df["hour"]         = df["time"].dt.hour
df["day_of_week"]  = df["time"].dt.day_name()
df["date"]         = df["time"].dt.date
df["is_daytime"]   = df["hour"].between(6, 20).astype(int)
df["feels_like_temp"] = (
    df["temperature_2m"] -
    0.4*(df["temperature_2m"]-10)*(1-df["relative_humidity_2m"]/100)
).round(2)
weather_map = {0:"Clear",1:"Mainly Clear",2:"Partly Cloudy",
               3:"Overcast",45:"Foggy",61:"Light Rain",80:"Showers"}
df["weather_label"] = df["weathercode"].map(weather_map).fillna("Unknown")
print("  ✅ hour, day_of_week, date, is_daytime added")
print("  ✅ feels_like_temp calculated")
print("  ✅ weather_label mapped from weathercode")

# ─────────────────────────────────────────────────────────────
# SECTION 8 — ENCODE CATEGORICAL VARIABLES
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 8: Encoding Categorical Variables")
print("=" * 60)

le = LabelEncoder()
df["day_of_week_encoded"] = le.fit_transform(df["day_of_week"])
print(f"  ✅ day_of_week label encoded. Classes: {list(le.classes_)}")
df = pd.get_dummies(df, columns=["weather_label"], prefix="weather")
print("  ✅ weather_label one-hot encoded → weather_* columns added")

# ─────────────────────────────────────────────────────────────
# SECTION 9 — SCALE NUMERIC FEATURES
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 9: Scaling Numeric Features")
print("=" * 60)

scale_cols = ["temperature_2m","relative_humidity_2m",
              "precipitation","windspeed_10m","feels_like_temp"]

df_std = pd.DataFrame(
    StandardScaler().fit_transform(df[scale_cols]),
    columns=[f"{c}_zscore" for c in scale_cols])

df_mm = pd.DataFrame(
    MinMaxScaler().fit_transform(df[scale_cols]),
    columns=[f"{c}_minmax" for c in scale_cols])

df = pd.concat([df, df_std, df_mm], axis=1)
print("  ✅ Z-score (StandardScaler) → *_zscore columns added")
print("  ✅ Min-Max (MinMaxScaler)   → *_minmax columns added")

# ─────────────────────────────────────────────────────────────
# SECTION 10 — EXPORT CLEANED DATA
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  STEP 10: Exporting Cleaned Dataset")
print("=" * 60)

df.drop(columns=["time","date","day_of_week"], errors="ignore")\
  .to_csv("lodi_nj_weather_cleaned.csv", index=False)
print("  ✅ Cleaned data saved → lodi_nj_weather_cleaned.csv")
print(f"  📐 Final shape: {df.shape[0]} rows x {df.shape[1]} columns")

# ─────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  ✅  PIPELINE COMPLETE — SUMMARY")
print("=" * 60)
for k, v in {
    "Total Rows"              : df.shape[0],
    "Total Columns"           : df.shape[1],
    "Missing Values Remaining": int(df.isnull().sum().sum()),
    "Features Engineered"     : 5,
    "Scalers Used"            : 2,
    "Output Files"            : "cleaned CSV + PNG chart",
}.items():
    print(f"  {k:<30}: {v}")
print("=" * 60)


  STEP 1: Fetching live weather data for Lodi, NJ...
  ✅ Data fetched! Shape: 360 rows x 6 columns

  STEP 2: Initial Data Exploration

📐 Shape      : 360 rows, 6 columns

📋 Columns    : ['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'windspeed_10m', 'weathercode']

🔢 Data Types :
time                        str
temperature_2m          float64
relative_humidity_2m      int64
precipitation           float64
windspeed_10m           float64
weathercode               int64
dtype: object

📊 First 5 rows:
               time  temperature_2m  relative_humidity_2m  precipitation  \
0  2026-05-17T00:00            20.9                    51            0.0   
1  2026-05-17T01:00            20.1                    54            0.0   
2  2026-05-17T02:00            19.9                    54            0.0   
3  2026-05-17T03:00            19.2                    62            0.0   
4  2026-05-17T04:00            18.7                    74            0.0   

   windspeed_10m 

In [ ]:
# ============================================================
#  STEP 11 — Machine Learning: Predict Rain vs. No Rain
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

df_ml = pd.read_csv('lodi_nj_weather_cleaned.csv')

# Use weathercode for rain label — more balanced than precipitation > 0
# Codes 51–82 cover drizzle, rain, showers
rain_codes = [51,53,55,61,63,65,71,73,75,80,81,82]
df_ml['rain'] = df_ml['weathercode'].isin(rain_codes).astype(int)

print("🌧️  Class balance:", df_ml['rain'].value_counts().to_dict())

features = [c for c in df_ml.columns if c.endswith('_zscore')]
X = df_ml[features].fillna(0)
y = df_ml['rain']

# No stratify — avoids error when one class is tiny
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print("\n📊 Classification Report:")
print(classification_report(y_test, rf.predict(X_test),
      labels=[0, 1], target_names=['No Rain', 'Rain'],
      zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
importances = pd.Series(rf.feature_importances_, index=features).sort_values()
importances.plot(kind='barh', ax=axes[0], color='#42A5F5', edgecolor='white')
axes[0].set_title('Feature Importances', fontweight='bold')

ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test,
    display_labels=['No Rain','Rain'], cmap='Blues', ax=axes[1])
axes[1].set_title('Confusion Matrix', fontweight='bold')

plt.suptitle('Random Forest — Rain Prediction (Lodi, NJ)', fontweight='bold')
plt.tight_layout()
plt.show()
print(f"✅ Model Accuracy: {rf.score(X_test, y_test):.1%}")



In [ ]:
# ============================================================
#  STEP 12 — 7-Day Rain Forecast for Lodi, NJ
# ============================================================
print("=" * 60)
print("  STEP 12: 7-Day Rain Prediction — Lodi, NJ")
print("=" * 60)

# 1) Re-train model on full cleaned dataset
df_hist = pd.read_csv('lodi_nj_weather_cleaned.csv')
rain_codes = [51,53,55,61,63,65,71,73,75,80,81,82]
df_hist['rain'] = df_hist['weathercode'].isin(rain_codes).astype(int)

feature_cols = ['temperature_2m', 'relative_humidity_2m',
                'precipitation', 'windspeed_10m', 'feels_like_temp']

scaler2 = StandardScaler()
X_hist = scaler2.fit_transform(df_hist[feature_cols].fillna(0))
rf2 = RandomForestClassifier(n_estimators=100, random_state=42)
rf2.fit(X_hist, df_hist['rain'])
print(f"  ✅ Model re-trained on {len(df_hist)} historical rows")

# 2) Fetch 7-day forecast from Open-Meteo
fc_url = "https://api.open-meteo.com/v1/forecast"
fc_params = {
    "latitude": LAT,
    "longitude": LON,
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,windspeed_10m,weathercode",
    "forecast_days": 7,
    "timezone": "America/New_York"
}
df_fc = pd.DataFrame(requests.get(fc_url, params=fc_params).json()['hourly'])
df_fc.rename(columns={"relative_humidity_2m": "relative_humidity_2m"}, inplace=True)
print(f"  ✅ Forecast fetched: {len(df_fc)} hourly rows (7 days)")

# 3) Engineer same feels_like_temp feature
df_fc['feels_like_temp'] = (
    df_fc['temperature_2m']
    - 0.4 * (df_fc['temperature_2m'] - 10)
    * (1 - df_fc['relative_humidity_2m'] / 100)
)

# 4) Scale and predict
X_fc = scaler2.transform(df_fc[feature_cols].fillna(0))
df_fc['rain_predicted'] = rf2.predict(X_fc)
df_fc['rain_prob'] = rf2.predict_proba(X_fc)[:, 1]

# 5) Daily summary table
df_fc['date'] = pd.to_datetime(df_fc['time']).dt.date
daily = df_fc.groupby('date').agg(
    rain_hours=('rain_predicted', 'sum'),
    max_prob=('rain_prob', 'max'),
    total_hours=('rain_predicted', 'count')
).reset_index()
daily['pct_hours'] = (daily['rain_hours'] / daily['total_hours'] * 100).round(1)
# simple flag: likely rain if any hour prob >= 0.5 or >=25% of hours predicted wet
daily['rain_likely'] = ((daily['max_prob'] >= 0.5) | (daily['pct_hours'] >= 25)).astype(int)

print("\n7-day summary:")
print(daily[['date', 'rain_hours', 'total_hours', 'pct_hours', 'max_prob', 'rain_likely']])

# 6) Quick visual
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
daily.plot(x='date', y='rain_hours', kind='bar', ax=ax, color='#2196F3', legend=False)
ax2 = ax.twinx()
daily.plot(x='date', y='max_prob', ax=ax2, color='#FF5722', marker='o', legend=False)
ax.set_ylabel('Predicted Rain Hours')
ax2.set_ylabel('Max Rain Probability')
plt.title('7-Day Rain Forecast Summary — Lodi, NJ')
plt.tight_layout()
plt.show()

  STEP 12: 7-Day Rain Prediction — Lodi, NJ
  ✅ Model re-trained on 360 historical rows
  ✅ Forecast fetched: 168 hourly rows (7 days)


KeyError: 'relative_humidity_2m'